# HW1 加分題：AHB / AXI Wrapper 的 RTL 模擬（Google Colab 版）

SIP：OpenCores **wb_dma**。本筆記本包含第 1 題四個 wrapper 的 Verilog、testbench 與行為模型，
會用 **Icarus Verilog** 編譯模擬，並畫出波形圖。

## 使用方式
1. 上方選單 **執行階段（Runtime）→ 全部執行（Run all）**，或按 `Ctrl + F9`
2. 等全部跑完（約 1～2 分鐘），往下看每個區塊的結果
3. 最後一格會自動下載 `hw1_bonus_results.zip`（程式碼＋模擬紀錄＋波形圖）

> 第一次執行若跳出「這個筆記本並非由 Google 建立」的警告，按 **仍要執行** 即可。

## 1. 安裝模擬器與中文字型

In [ ]:
# 安裝 Icarus Verilog（模擬器）與 Noto CJK 中文字型（波形圖標題用）
!apt-get -qq update > /dev/null
!apt-get -qq install -y iverilog fonts-noto-cjk > /dev/null
!rm -rf ~/.cache/matplotlib
!iverilog -V 2>&1 | head -1

# 建立專案資料夾
import os
os.makedirs('/content/hw1_bonus', exist_ok=True)
%cd /content/hw1_bonus
!mkdir -p rtl tb sim wave scripts

## 2. RTL：四個 wrapper（加分題的主角）
每個檔案的狀態名稱與轉移編號（①②③…）都對應第 1 題的 FSM 狀態圖。
執行後會在 `rtl/` 產生檔案；想看程式碼可以展開下面各格。

In [ ]:
%%writefile rtl/ahb_slave_wrapper.v
// ============================================================================
// AHB Slave Wrapper  (AHB 協定 -> Wishbone)
// ----------------------------------------------------------------------------
// 用途：CPU（AHB master）讀寫 wb_dma 的控制暫存器時，
//       本 wrapper 接收 AHB 傳輸，轉成 Wishbone classic cycle 送給 wb_dma 的
//       Wishbone slave 介面。
//
// FSM（對應作業第 3 步的狀態圖）：
//   IDLE      : HREADYOUT=1，等待有效傳輸 (HSEL & HREADY & HTRANS=NONSEQ/SEQ)
//   WB_ACCESS : 用鎖存的位址發出 Wishbone 傳輸，HREADYOUT=0 插入等待
//   ERR1      : 錯誤回應第 1 拍 (HRESP=ERROR, HREADYOUT=0)
//   ERR2      : 錯誤回應第 2 拍 (HRESP=ERROR, HREADYOUT=1)
// ============================================================================
module ahb_slave_wrapper (
    input  wire        HCLK,
    input  wire        HRESETn,
    // ---------------- AHB slave 介面 ----------------
    input  wire        HSEL,
    input  wire [31:0] HADDR,
    input  wire        HWRITE,
    input  wire [1:0]  HTRANS,
    input  wire [2:0]  HSIZE,
    input  wire [31:0] HWDATA,
    input  wire        HREADY,      // 匯流排上的 HREADY（上一筆是否完成）
    output wire        HREADYOUT,   // 本 slave 回給匯流排的 ready
    output wire [1:0]  HRESP,
    output wire [31:0] HRDATA,
    // ---------------- Wishbone master 介面（接 wb_dma 的 slave 埠） -------
    output wire        wb_cyc_o,
    output wire        wb_stb_o,
    output wire        wb_we_o,
    output wire [31:0] wb_adr_o,
    output wire [3:0]  wb_sel_o,
    output wire [31:0] wb_dat_o,
    input  wire [31:0] wb_dat_i,
    input  wire        wb_ack_i,
    input  wire        wb_err_i
);

    // HTRANS / HRESP 編碼
    localparam HTRANS_IDLE   = 2'b00,
               HTRANS_BUSY   = 2'b01,
               HTRANS_NONSEQ = 2'b10,
               HTRANS_SEQ    = 2'b11;
    localparam HRESP_OKAY    = 2'b00,
               HRESP_ERROR   = 2'b01;

    // 狀態編碼
    localparam S_IDLE      = 2'd0,
               S_WB_ACCESS = 2'd1,
               S_ERR1      = 2'd2,
               S_ERR2      = 2'd3;

    reg [1:0]  state, next_state;

    // 位址相位鎖存的控制訊號（AHB 是 pipeline，資料相位時 HADDR 已經換了）
    reg [31:0] addr_q;
    reg        write_q;
    reg [3:0]  sel_q;
    reg [31:0] rdata_q;

    // 「有效傳輸」= 被選中 & 匯流排上一筆已完成 & 傳輸型態是 NONSEQ 或 SEQ
    wire valid_xfer = HSEL & HREADY & HTRANS[1];

    // 依 HSIZE 與 HADDR[1:0] 產生 byte select
    function [3:0] size2sel;
        input [2:0] size;
        input [1:0] a;
        begin
            case (size)
                3'b000:  size2sel = 4'b0001 << a;                 // byte
                3'b001:  size2sel = a[1] ? 4'b1100 : 4'b0011;     // halfword
                default: size2sel = 4'b1111;                      // word
            endcase
        end
    endfunction

    // ---------------- 狀態暫存器 ----------------
    always @(posedge HCLK or negedge HRESETn) begin
        if (!HRESETn) state <= S_IDLE;
        else          state <= next_state;
    end

    // ---------------- 次態邏輯 ----------------
    always @(*) begin
        next_state = state;
        case (state)
            S_IDLE:      if (valid_xfer)    next_state = S_WB_ACCESS;   // ①
            S_WB_ACCESS: if (wb_ack_i)      next_state = S_IDLE;        // ②
                         else if (wb_err_i) next_state = S_ERR1;        // ⑤
            S_ERR1:                         next_state = S_ERR2;        // ⑥
            S_ERR2:      if (valid_xfer)    next_state = S_WB_ACCESS;   // ⑧
                         else               next_state = S_IDLE;        // ⑦
            default:                        next_state = S_IDLE;
        endcase
    end

    // ---------------- 資料路徑：鎖存位址 / 讀回資料 ----------------
    always @(posedge HCLK or negedge HRESETn) begin
        if (!HRESETn) begin
            addr_q  <= 32'd0;
            write_q <= 1'b0;
            sel_q   <= 4'd0;
            rdata_q <= 32'd0;
        end else begin
            // 位址相位：在 IDLE / ERR2（HREADYOUT=1 的狀態）取樣新傳輸
            if ((state == S_IDLE || state == S_ERR2) && valid_xfer) begin
                addr_q  <= HADDR;
                write_q <= HWRITE;
                sel_q   <= size2sel(HSIZE, HADDR[1:0]);
            end
            // Wishbone 回 ACK 時把讀到的資料存起來，下一拍 (IDLE) 送上 HRDATA
            if (state == S_WB_ACCESS && wb_ack_i && !write_q)
                rdata_q <= wb_dat_i;
        end
    end

    // ---------------- 輸出（Moore 型） ----------------
    assign HREADYOUT = (state == S_IDLE) || (state == S_ERR2);
    assign HRESP     = (state == S_ERR1 || state == S_ERR2) ? HRESP_ERROR : HRESP_OKAY;
    assign HRDATA    = rdata_q;

    assign wb_cyc_o  = (state == S_WB_ACCESS);
    assign wb_stb_o  = (state == S_WB_ACCESS);
    assign wb_we_o   = write_q;
    assign wb_adr_o  = addr_q;
    assign wb_sel_o  = sel_q;
    // HREADYOUT=0 期間 AHB master 必須保持 HWDATA，所以可以直接接線
    assign wb_dat_o  = HWDATA;

endmodule

In [ ]:
%%writefile rtl/ahb_master_wrapper.v
// ============================================================================
// AHB Master Wrapper  (Wishbone -> AHB 協定，含 AHB2 仲裁)
// ----------------------------------------------------------------------------
// 用途：wb_dma 的 DMA 引擎用 Wishbone master 發出讀寫，本 wrapper 先向
//       AHB 仲裁器申請匯流排 (HBUSREQ/HGRANT)，再轉成 AHB 單筆傳輸。
//
// FSM（對應作業第 4 步的狀態圖）：
//   IDLE    : 不申請匯流排，等 Wishbone 請求
//   BUS_REQ : HBUSREQ=1，等 HGRANT & HREADY
//   ADDR    : 送出位址相位 (HTRANS=NONSEQ)，等 HREADY
//   DATA    : 資料相位 (HTRANS=IDLE)，等 HREADY 後依 HRESP 決定結果
// ============================================================================
module ahb_master_wrapper (
    input  wire        HCLK,
    input  wire        HRESETn,
    // ---------------- AHB master 介面 ----------------
    output wire        HBUSREQ,
    input  wire        HGRANT,
    input  wire        HREADY,
    input  wire [1:0]  HRESP,
    input  wire [31:0] HRDATA,
    output wire [31:0] HADDR,
    output wire [1:0]  HTRANS,
    output wire        HWRITE,
    output wire [2:0]  HSIZE,
    output wire [2:0]  HBURST,
    output wire [31:0] HWDATA,
    // ---------------- Wishbone slave 介面（接 wb_dma 的 master 埠） -------
    input  wire        wb_cyc_i,
    input  wire        wb_stb_i,
    input  wire        wb_we_i,
    input  wire [31:0] wb_adr_i,
    input  wire [3:0]  wb_sel_i,
    input  wire [31:0] wb_dat_i,
    output wire [31:0] wb_dat_o,
    output wire        wb_ack_o,
    output wire        wb_err_o
);

    localparam HTRANS_IDLE   = 2'b00,
               HTRANS_NONSEQ = 2'b10;
    localparam HRESP_OKAY    = 2'b00,
               HRESP_ERROR   = 2'b01,
               HRESP_RETRY   = 2'b10,
               HRESP_SPLIT   = 2'b11;
    localparam HBURST_SINGLE = 3'b000;

    localparam S_IDLE    = 2'd0,
               S_BUS_REQ = 2'd1,
               S_ADDR    = 2'd2,
               S_DATA    = 2'd3;

    reg [1:0] state, next_state;

    wire wb_req = wb_cyc_i & wb_stb_i;

    // ---------------- 狀態暫存器 ----------------
    always @(posedge HCLK or negedge HRESETn) begin
        if (!HRESETn) state <= S_IDLE;
        else          state <= next_state;
    end

    // ---------------- 次態邏輯 ----------------
    always @(*) begin
        next_state = state;
        case (state)
            S_IDLE:    if (wb_req)            next_state = S_BUS_REQ;   // ①
            S_BUS_REQ: if (HGRANT && HREADY)  next_state = S_ADDR;      // ② 取得授權
            S_ADDR:    if (HREADY)            next_state = S_DATA;      // ③ 位址被接受
            S_DATA:    if (HREADY) begin
                           if (HRESP == HRESP_RETRY || HRESP == HRESP_SPLIT)
                                              next_state = S_BUS_REQ;   // ⑤ 重新申請
                           else               next_state = S_IDLE;      // ④ OKAY / ERROR
                       end
            default:                          next_state = S_IDLE;
        endcase
    end

    // 由 Wishbone 的 byte select 推出 HSIZE 與位址低兩位
    reg [2:0] size_c;
    reg [1:0] alow_c;
    always @(*) begin
        case (wb_sel_i)
            4'b0001: begin size_c = 3'b000; alow_c = 2'd0; end
            4'b0010: begin size_c = 3'b000; alow_c = 2'd1; end
            4'b0100: begin size_c = 3'b000; alow_c = 2'd2; end
            4'b1000: begin size_c = 3'b000; alow_c = 2'd3; end
            4'b0011: begin size_c = 3'b001; alow_c = 2'd0; end
            4'b1100: begin size_c = 3'b001; alow_c = 2'd2; end
            default: begin size_c = 3'b010; alow_c = 2'd0; end
        endcase
    end

    // ---------------- AHB 輸出 ----------------
    // Wishbone master 會保持位址與資料直到 ACK，所以這裡直接接線，不需鎖存
    assign HBUSREQ = (state == S_BUS_REQ) || (state == S_ADDR);
    assign HTRANS  = (state == S_ADDR) ? HTRANS_NONSEQ : HTRANS_IDLE;
    assign HADDR   = {wb_adr_i[31:2], alow_c};
    assign HWRITE  = wb_we_i;
    assign HSIZE   = size_c;
    assign HBURST  = HBURST_SINGLE;
    assign HWDATA  = wb_dat_i;

    // ---------------- Wishbone 輸出 ----------------
    assign wb_dat_o = HRDATA;
    assign wb_ack_o = (state == S_DATA) && HREADY && (HRESP == HRESP_OKAY);
    assign wb_err_o = (state == S_DATA) && HREADY && (HRESP == HRESP_ERROR);

endmodule

In [ ]:
%%writefile rtl/axi_slave_wrapper.v
// ============================================================================
// AXI4 Slave Wrapper  (AXI4 協定 -> Wishbone)
// ----------------------------------------------------------------------------
// 用途：CPU（AXI master）讀寫 wb_dma 的控制暫存器時，本 wrapper 接收 AXI
//       傳輸（支援 burst），逐拍轉成 Wishbone classic cycle。
//
// 因為 wb_dma 的 Wishbone slave 只有一個埠，讀寫必須序列化，
// 所以讀、寫兩條路徑共用一個 IDLE，兩邊同時有請求時寫入優先。
//
// FSM（對應作業第 5 步的狀態圖）：
//   IDLE   : AWREADY=1；AWVALID=0 時 ARREADY=1
//   W_DATA : WVALID=1 時發 Wishbone 寫入，ACK 回來才拉 WREADY
//   W_RESP : BVALID=1 送寫入回應
//   R_WB   : 發 Wishbone 讀取，等 ACK
//   R_DATA : RVALID=1 送讀取資料，最後一拍 RLAST=1
//
// 簡化假設：資料寬度 32 bit、支援 FIXED/INCR burst（WRAP 當 INCR 處理）、
//          一次只處理一筆交易（不支援 outstanding）。
// ============================================================================
module axi_slave_wrapper #(
    parameter ID_W = 4
)(
    input  wire            ACLK,
    input  wire            ARESETn,
    // ---------------- AXI 寫入位址通道 AW ----------------
    input  wire [ID_W-1:0] AWID,
    input  wire [31:0]     AWADDR,
    input  wire [7:0]      AWLEN,
    input  wire [2:0]      AWSIZE,
    input  wire [1:0]      AWBURST,
    input  wire            AWVALID,
    output wire            AWREADY,
    // ---------------- AXI 寫入資料通道 W ----------------
    input  wire [31:0]     WDATA,
    input  wire [3:0]      WSTRB,
    input  wire            WLAST,
    input  wire            WVALID,
    output wire            WREADY,
    // ---------------- AXI 寫入回應通道 B ----------------
    output wire [ID_W-1:0] BID,
    output wire [1:0]      BRESP,
    output wire            BVALID,
    input  wire            BREADY,
    // ---------------- AXI 讀取位址通道 AR ----------------
    input  wire [ID_W-1:0] ARID,
    input  wire [31:0]     ARADDR,
    input  wire [7:0]      ARLEN,
    input  wire [2:0]      ARSIZE,
    input  wire [1:0]      ARBURST,
    input  wire            ARVALID,
    output wire            ARREADY,
    // ---------------- AXI 讀取資料通道 R ----------------
    output wire [ID_W-1:0] RID,
    output wire [31:0]     RDATA,
    output wire [1:0]      RRESP,
    output wire            RLAST,
    output wire            RVALID,
    input  wire            RREADY,
    // ---------------- Wishbone master 介面（接 wb_dma 的 slave 埠） -------
    output wire            wb_cyc_o,
    output wire            wb_stb_o,
    output wire            wb_we_o,
    output wire [31:0]     wb_adr_o,
    output wire [3:0]      wb_sel_o,
    output wire [31:0]     wb_dat_o,
    input  wire [31:0]     wb_dat_i,
    input  wire            wb_ack_i,
    input  wire            wb_err_i
);

    localparam RESP_OKAY   = 2'b00,
               RESP_SLVERR = 2'b10;
    localparam BURST_FIXED = 2'b00;

    localparam S_IDLE   = 3'd0,
               S_W_DATA = 3'd1,
               S_W_RESP = 3'd2,
               S_R_WB   = 3'd3,
               S_R_DATA = 3'd4;

    reg [2:0]      state, next_state;

    reg [31:0]     addr_q;      // 目前這一拍的位址（burst 時由 wrapper 自己遞增）
    reg [7:0]      cnt_q;       // 讀取 burst 剩餘拍數
    reg [2:0]      size_q;
    reg [1:0]      burst_q;
    reg [ID_W-1:0] id_q;
    reg [1:0]      bresp_q;     // 整個寫入 burst 的累積回應
    reg [31:0]     rdata_q;
    reg [1:0]      rresp_q;

    // 握手成立的條件（VALID 與 READY 同一個時脈邊緣都為 1）
    wire aw_hs   = AWVALID & AWREADY;
    wire ar_hs   = ARVALID & ARREADY;
    wire w_done  = (state == S_W_DATA) & WVALID & (wb_ack_i | wb_err_i);
    wire r_wbdone= (state == S_R_WB) & (wb_ack_i | wb_err_i);

    // 下一拍位址：FIXED 不變，INCR 加上 2^SIZE
    wire [31:0] addr_next = (burst_q == BURST_FIXED) ? addr_q
                                                     : addr_q + (32'd1 << size_q);

    // ---------------- 狀態暫存器 ----------------
    always @(posedge ACLK or negedge ARESETn) begin
        if (!ARESETn) state <= S_IDLE;
        else          state <= next_state;
    end

    // ---------------- 次態邏輯 ----------------
    always @(*) begin
        next_state = state;
        case (state)
            S_IDLE:   if (aw_hs)       next_state = S_W_DATA;                 // ①
                      else if (ar_hs)  next_state = S_R_WB;                   // ⑤
            S_W_DATA: if (w_done && WLAST) next_state = S_W_RESP;             // ③（②為自迴圈）
            S_W_RESP: if (BREADY)      next_state = S_IDLE;                   // ④
            S_R_WB:   if (r_wbdone)    next_state = S_R_DATA;                 // ⑥
            S_R_DATA: if (RREADY)      next_state = (cnt_q == 8'd0) ? S_IDLE  // ⑦
                                                                    : S_R_WB; // ⑧
            default:                   next_state = S_IDLE;
        endcase
    end

    // ---------------- 資料路徑 ----------------
    always @(posedge ACLK or negedge ARESETn) begin
        if (!ARESETn) begin
            addr_q  <= 32'd0;  cnt_q   <= 8'd0;  size_q  <= 3'd2;
            burst_q <= 2'b01;  id_q    <= {ID_W{1'b0}};
            bresp_q <= RESP_OKAY;
            rdata_q <= 32'd0;  rresp_q <= RESP_OKAY;
        end else begin
            case (state)
                S_IDLE: begin
                    if (aw_hs) begin                 // 鎖存寫入位址與 ID
                        addr_q  <= AWADDR;  size_q <= AWSIZE;
                        burst_q <= AWBURST; id_q   <= AWID;
                        bresp_q <= RESP_OKAY;
                    end else if (ar_hs) begin        // 鎖存讀取位址、長度與 ID
                        addr_q  <= ARADDR;  size_q <= ARSIZE;
                        burst_q <= ARBURST; id_q   <= ARID;
                        cnt_q   <= ARLEN;
                    end
                end
                S_W_DATA: if (w_done) begin
                    addr_q <= addr_next;             // ② 下一拍位址
                    if (wb_err_i) bresp_q <= RESP_SLVERR;  // 出錯也要把整個 burst 收完
                end
                S_R_WB: if (r_wbdone) begin
                    rdata_q <= wb_dat_i;
                    rresp_q <= wb_err_i ? RESP_SLVERR : RESP_OKAY;
                end
                S_R_DATA: if (RREADY && cnt_q != 8'd0) begin   // ⑧
                    cnt_q  <= cnt_q - 8'd1;
                    addr_q <= addr_next;
                end
                default: ;
            endcase
        end
    end

    // ---------------- AXI 輸出 ----------------
    assign AWREADY = (state == S_IDLE);
    assign ARREADY = (state == S_IDLE) & ~AWVALID;    // 寫入優先
    assign WREADY  = w_done;                          // ACK 回來才吃下這拍
    assign BVALID  = (state == S_W_RESP);
    assign BRESP   = bresp_q;
    assign BID     = id_q;
    assign RVALID  = (state == S_R_DATA);
    assign RDATA   = rdata_q;
    assign RRESP   = rresp_q;
    assign RLAST   = (state == S_R_DATA) & (cnt_q == 8'd0);
    assign RID     = id_q;

    // ---------------- Wishbone 輸出 ----------------
    assign wb_cyc_o = ((state == S_W_DATA) & WVALID) | (state == S_R_WB);
    assign wb_stb_o = wb_cyc_o;
    assign wb_we_o  = (state == S_W_DATA);
    assign wb_adr_o = addr_q;
    assign wb_sel_o = (state == S_W_DATA) ? WSTRB : 4'b1111;
    assign wb_dat_o = WDATA;    // WREADY=0 期間 AXI master 必須保持 WDATA

endmodule

In [ ]:
%%writefile rtl/axi_master_wrapper.v
// ============================================================================
// AXI4 Master Wrapper  (Wishbone -> AXI4 協定)
// ----------------------------------------------------------------------------
// 用途：wb_dma 的 DMA 引擎用 Wishbone master 發出讀寫，本 wrapper 依 WE
//       走寫入通道 (AW + W + B) 或讀取通道 (AR + R)。
//
// 重點：AXI 規範禁止 master 等 AWREADY 才拉 WVALID（slave 可能先等 WVALID
//       才給 AWREADY，互等就死結），所以 W_SEND 狀態同時拉高 AWVALID 與
//       WVALID，用 aw_done / w_done 兩個旗標記錄誰先完成。
//
// FSM（對應作業第 6 步的狀態圖）：
//   IDLE   : 等 Wishbone 請求，依 WE 分流
//   W_SEND : AWVALID、WVALID 同時拉高，兩個握手都完成才離開
//   W_RESP : BREADY=1，等 BVALID
//   R_ADDR : ARVALID=1，等 ARREADY
//   R_DATA : RREADY=1，等 RVALID
// ============================================================================
module axi_master_wrapper #(
    parameter ID_W = 4
)(
    input  wire            ACLK,
    input  wire            ARESETn,
    // ---------------- AXI 寫入位址通道 AW ----------------
    output wire [ID_W-1:0] AWID,
    output wire [31:0]     AWADDR,
    output wire [7:0]      AWLEN,
    output wire [2:0]      AWSIZE,
    output wire [1:0]      AWBURST,
    output wire            AWVALID,
    input  wire            AWREADY,
    // ---------------- AXI 寫入資料通道 W ----------------
    output wire [31:0]     WDATA,
    output wire [3:0]      WSTRB,
    output wire            WLAST,
    output wire            WVALID,
    input  wire            WREADY,
    // ---------------- AXI 寫入回應通道 B ----------------
    input  wire [ID_W-1:0] BID,
    input  wire [1:0]      BRESP,
    input  wire            BVALID,
    output wire            BREADY,
    // ---------------- AXI 讀取位址通道 AR ----------------
    output wire [ID_W-1:0] ARID,
    output wire [31:0]     ARADDR,
    output wire [7:0]      ARLEN,
    output wire [2:0]      ARSIZE,
    output wire [1:0]      ARBURST,
    output wire            ARVALID,
    input  wire            ARREADY,
    // ---------------- AXI 讀取資料通道 R ----------------
    input  wire [ID_W-1:0] RID,
    input  wire [31:0]     RDATA,
    input  wire [1:0]      RRESP,
    input  wire            RLAST,
    input  wire            RVALID,
    output wire            RREADY,
    // ---------------- Wishbone slave 介面（接 wb_dma 的 master 埠） -------
    input  wire            wb_cyc_i,
    input  wire            wb_stb_i,
    input  wire            wb_we_i,
    input  wire [31:0]     wb_adr_i,
    input  wire [3:0]      wb_sel_i,
    input  wire [31:0]     wb_dat_i,
    output wire [31:0]     wb_dat_o,
    output wire            wb_ack_o,
    output wire            wb_err_o
);

    localparam S_IDLE   = 3'd0,
               S_W_SEND = 3'd1,
               S_W_RESP = 3'd2,
               S_R_ADDR = 3'd3,
               S_R_DATA = 3'd4;

    reg [2:0] state, next_state;
    reg       aw_done, w_done;

    wire wb_req = wb_cyc_i & wb_stb_i;
    wire aw_hs  = AWVALID & AWREADY;
    wire w_hs   = WVALID  & WREADY;
    // AW、W 兩個握手都完成（可以是之前完成的，也可以是這一拍同時完成）
    wire both_done = (aw_done | aw_hs) & (w_done | w_hs);

    // ---------------- 狀態暫存器 ----------------
    always @(posedge ACLK or negedge ARESETn) begin
        if (!ARESETn) state <= S_IDLE;
        else          state <= next_state;
    end

    // ---------------- 次態邏輯 ----------------
    always @(*) begin
        next_state = state;
        case (state)
            S_IDLE:   if (wb_req)    next_state = wb_we_i ? S_W_SEND   // ①
                                                          : S_R_ADDR;  // ④
            S_W_SEND: if (both_done) next_state = S_W_RESP;            // ②
            S_W_RESP: if (BVALID)    next_state = S_IDLE;              // ③
            S_R_ADDR: if (ARREADY)   next_state = S_R_DATA;            // ⑤
            S_R_DATA: if (RVALID)    next_state = S_IDLE;              // ⑥
            default:                 next_state = S_IDLE;
        endcase
    end

    // ---------------- aw_done / w_done 旗標 ----------------
    always @(posedge ACLK or negedge ARESETn) begin
        if (!ARESETn) begin
            aw_done <= 1'b0;
            w_done  <= 1'b0;
        end else if (state == S_W_SEND) begin
            if (aw_hs) aw_done <= 1'b1;
            if (w_hs)  w_done  <= 1'b1;
        end else begin
            aw_done <= 1'b0;   // 離開 W_SEND 後清除，下一筆重新開始
            w_done  <= 1'b0;
        end
    end

    // ---------------- AXI 輸出 ----------------
    // Wishbone 會保持位址與資料直到 ACK，所以直接接線；也自然滿足
    // 「VALID 拉高後在握手前內容不能變」的 AXI 規則
    assign AWID    = {ID_W{1'b0}};
    assign AWADDR  = wb_adr_i;
    assign AWLEN   = 8'd0;          // 單拍
    assign AWSIZE  = 3'b010;        // 4 bytes
    assign AWBURST = 2'b01;         // INCR
    assign AWVALID = (state == S_W_SEND) & ~aw_done;

    assign WDATA   = wb_dat_i;
    assign WSTRB   = wb_sel_i;
    assign WLAST   = 1'b1;
    assign WVALID  = (state == S_W_SEND) & ~w_done;

    assign BREADY  = (state == S_W_RESP);

    assign ARID    = {ID_W{1'b0}};
    assign ARADDR  = wb_adr_i;
    assign ARLEN   = 8'd0;
    assign ARSIZE  = 3'b010;
    assign ARBURST = 2'b01;
    assign ARVALID = (state == S_R_ADDR);

    assign RREADY  = (state == S_R_DATA);

    // ---------------- Wishbone 輸出 ----------------
    // RESP[1]=0 代表 OKAY/EXOKAY，RESP[1]=1 代表 SLVERR/DECERR
    assign wb_dat_o = RDATA;
    assign wb_ack_o = ((state == S_W_RESP) & BVALID & ~BRESP[1]) |
                      ((state == S_R_DATA) & RVALID & ~RRESP[1]);
    assign wb_err_o = ((state == S_W_RESP) & BVALID &  BRESP[1]) |
                      ((state == S_R_DATA) & RVALID &  RRESP[1]);

endmodule

## 3. Testbench 與行為模型
- `wb_slave_model.v`：模擬 wb_dma 的控制暫存器（Wishbone slave）
- `ahb_mem_model.v`：AHB 記憶體控制器，含 ERROR 區與 RETRY 區
- `axi_mem_model.v`：AXI 記憶體控制器，**故意先收 W 才給 AWREADY**（用來驗證 master 不會死結）

In [ ]:
%%writefile tb/wb_slave_model.v
// ============================================================================
// Wishbone slave 行為模型：模擬 wb_dma 的控制暫存器（slave 埠）
// ----------------------------------------------------------------------------
// - 64 個 32-bit 暫存器，位址 0x000 ~ 0x0FF
// - 位址 >= 0x100 視為不存在的暫存器，回 ERR
// - 參數 WAIT：收到 STB 後要等幾拍才回 ACK（模擬存取延遲）
// ============================================================================
module wb_slave_model #(
    parameter WAIT = 1
)(
    input  wire        clk,
    input  wire        rst,
    input  wire        wb_cyc_i,
    input  wire        wb_stb_i,
    input  wire        wb_we_i,
    input  wire [31:0] wb_adr_i,
    input  wire [3:0]  wb_sel_i,
    input  wire [31:0] wb_dat_i,
    output reg  [31:0] wb_dat_o,
    output reg         wb_ack_o,
    output reg         wb_err_o
);
    reg [31:0] regs [0:63];
    reg [3:0]  cnt;
    integer i;

    wire       hit = (wb_adr_i < 32'h100);
    wire [5:0] idx = wb_adr_i[7:2];

    always @(posedge clk) begin
        if (rst) begin
            wb_ack_o <= 1'b0;
            wb_err_o <= 1'b0;
            wb_dat_o <= 32'd0;
            cnt      <= 4'd0;
            for (i = 0; i < 64; i = i + 1) regs[i] <= 32'd0;
        end else begin
            wb_ack_o <= 1'b0;
            wb_err_o <= 1'b0;
            if (wb_cyc_i && wb_stb_i && !wb_ack_o && !wb_err_o) begin
                if (cnt == WAIT) begin
                    cnt <= 4'd0;
                    if (hit) begin
                        wb_ack_o <= 1'b1;
                        if (wb_we_i) begin
                            if (wb_sel_i[0]) regs[idx][7:0]   <= wb_dat_i[7:0];
                            if (wb_sel_i[1]) regs[idx][15:8]  <= wb_dat_i[15:8];
                            if (wb_sel_i[2]) regs[idx][23:16] <= wb_dat_i[23:16];
                            if (wb_sel_i[3]) regs[idx][31:24] <= wb_dat_i[31:24];
                        end else begin
                            wb_dat_o <= regs[idx];
                        end
                    end else begin
                        wb_err_o <= 1'b1;
                    end
                end else begin
                    cnt <= cnt + 4'd1;
                end
            end else begin
                cnt <= 4'd0;
            end
        end
    end
endmodule

In [ ]:
%%writefile tb/ahb_mem_model.v
// ============================================================================
// AHB slave 行為模型：模擬記憶體控制器
// ----------------------------------------------------------------------------
// - 256 words 記憶體（位址取 [9:2]）
// - 參數 WAIT：每筆資料相位插入的等待拍數
// - 位址 0x8000_0000 以上：回 ERROR（兩拍回應）
// - 位址 0x4000_0000 ~ 0x7FFF_FFFF：第一次存取回 RETRY（兩拍回應），
//   master 重送後才回 OKAY，用來測試 master wrapper 的 RETRY 處理
// ============================================================================
module ahb_mem_model #(
    parameter WAIT = 1
)(
    input  wire        HCLK,
    input  wire        HRESETn,
    input  wire        HSEL,
    input  wire [31:0] HADDR,
    input  wire        HWRITE,
    input  wire [1:0]  HTRANS,
    input  wire [2:0]  HSIZE,
    input  wire [31:0] HWDATA,
    input  wire        HREADY,
    output wire        HREADYOUT,
    output wire [1:0]  HRESP,
    output wire [31:0] HRDATA
);
    localparam OKAY = 2'b00, ERROR = 2'b01, RETRY = 2'b10;
    localparam ST_OK = 2'd0, ST_WAIT = 2'd1, ST_R1 = 2'd2, ST_R2 = 2'd3;

    reg [31:0] mem [0:255];
    reg [1:0]  st;
    reg [3:0]  cnt;
    reg        dp_active, dp_write;
    reg [31:0] dp_addr;
    reg [1:0]  kind;           // 這筆傳輸最後要回的 HRESP
    reg        retry_armed;    // 1 = 下一次存取 RETRY 區會被 RETRY
    integer i;

    wire new_xfer = HSEL & HREADY & HTRANS[1];

    function [1:0] classify;
        input [31:0] a;
        begin
            if (a[31])                      classify = ERROR;
            else if (a[30] && retry_armed)  classify = RETRY;
            else                            classify = OKAY;
        end
    endfunction

    always @(posedge HCLK or negedge HRESETn) begin
        if (!HRESETn) begin
            st <= ST_OK; cnt <= 0; dp_active <= 0; dp_write <= 0;
            dp_addr <= 0; kind <= OKAY; retry_armed <= 1'b1;
            for (i = 0; i < 256; i = i + 1) mem[i] <= 32'h0;
        end else begin
            // 資料相位在 HREADYOUT=1 的狀態（ST_OK / ST_R2）完成
            if ((st == ST_OK) && dp_active && dp_write && kind == OKAY)
                mem[dp_addr[9:2]] <= HWDATA;

            case (st)
                ST_OK, ST_R2: begin
                    if (new_xfer) begin
                        dp_active <= 1'b1;
                        dp_write  <= HWRITE;
                        dp_addr   <= HADDR;
                        kind      <= classify(HADDR);
                        if (classify(HADDR) == RETRY) retry_armed <= 1'b0;
                        else if (HADDR[30])           retry_armed <= 1'b1;
                        if (classify(HADDR) != OKAY)  st <= ST_R1;
                        else if (WAIT != 0) begin st <= ST_WAIT; cnt <= WAIT - 1; end
                        else                          st <= ST_OK;
                    end else begin
                        dp_active <= 1'b0;
                        st        <= ST_OK;
                    end
                end
                ST_WAIT: begin
                    if (cnt == 0) st <= ST_OK;
                    else          cnt <= cnt - 1;
                end
                ST_R1: st <= ST_R2;
                default: st <= ST_OK;
            endcase
        end
    end

    assign HREADYOUT = (st == ST_OK) || (st == ST_R2);
    assign HRESP     = (st == ST_R1 || st == ST_R2) ? kind : OKAY;
    assign HRDATA    = mem[dp_addr[9:2]];
endmodule

In [ ]:
%%writefile tb/axi_mem_model.v
// ============================================================================
// AXI4 slave 行為模型：模擬記憶體控制器（單拍傳輸）
// ----------------------------------------------------------------------------
// - 256 words 記憶體（位址取 [9:2]）
// - 故意設計成「先收完 W 才給 AWREADY」：這是 AXI 規範允許 slave 做的事，
//   如果 master 等 AWREADY 才送 WVALID，就會在這裡死結。用來驗證 master
//   wrapper 有同時送出 AW 與 W。
// - B、R 回應都會延遲 1 拍才給，展示 master 等待 VALID 的行為
// - 位址 0x8000_0000 以上回 SLVERR
// ============================================================================
module axi_mem_model (
    input  wire        ACLK,
    input  wire        ARESETn,
    input  wire [31:0] AWADDR,
    input  wire        AWVALID,
    output reg         AWREADY,
    input  wire [31:0] WDATA,
    input  wire [3:0]  WSTRB,
    input  wire        WVALID,
    output reg         WREADY,
    output reg  [1:0]  BRESP,
    output reg         BVALID,
    input  wire        BREADY,
    input  wire [31:0] ARADDR,
    input  wire        ARVALID,
    output reg         ARREADY,
    output reg  [31:0] RDATA,
    output reg  [1:0]  RRESP,
    output reg         RLAST,
    output reg         RVALID,
    input  wire        RREADY
);
    reg [31:0] mem [0:255];
    reg [31:0] waddr, raddr;
    reg        aw_got, w_got, b_pend, r_pend;
    reg [31:0] wdata_q;
    reg [3:0]  wstrb_q;
    integer i;

    always @(posedge ACLK or negedge ARESETn) begin
        if (!ARESETn) begin
            AWREADY <= 0; WREADY <= 0; BVALID <= 0; BRESP <= 0;
            ARREADY <= 0; RVALID <= 0; RDATA <= 0; RRESP <= 0; RLAST <= 0;
            aw_got <= 0; w_got <= 0; b_pend <= 0; r_pend <= 0;
            waddr <= 0; raddr <= 0; wdata_q <= 0; wstrb_q <= 0;
            for (i = 0; i < 256; i = i + 1) mem[i] <= 32'h0;
        end else begin
            // ---------------- 寫入 ----------------
            // 先收寫入資料，W 握手完成後才給 AWREADY（slave 合法行為）。
            // 若 master 等 AWREADY 才送 WVALID，這裡就會永遠等不到 -> 死結
            WREADY  <= !w_got  && !WREADY  && WVALID;
            AWREADY <= !aw_got && !AWREADY && AWVALID && w_got;
            if (AWVALID && AWREADY) begin aw_got <= 1; waddr <= AWADDR; end
            if (WVALID  && WREADY)  begin w_got  <= 1; wdata_q <= WDATA; wstrb_q <= WSTRB; end

            if (aw_got && w_got && !b_pend && !BVALID) begin
                b_pend <= 1;                       // 延遲一拍再送 B
                if (!waddr[31]) begin
                    if (wstrb_q[0]) mem[waddr[9:2]][7:0]   <= wdata_q[7:0];
                    if (wstrb_q[1]) mem[waddr[9:2]][15:8]  <= wdata_q[15:8];
                    if (wstrb_q[2]) mem[waddr[9:2]][23:16] <= wdata_q[23:16];
                    if (wstrb_q[3]) mem[waddr[9:2]][31:24] <= wdata_q[31:24];
                end
            end
            if (b_pend) begin
                b_pend <= 0;
                BVALID <= 1;
                BRESP  <= waddr[31] ? 2'b10 : 2'b00;
            end
            if (BVALID && BREADY) begin
                BVALID <= 0; aw_got <= 0; w_got <= 0;
            end

            // ---------------- 讀取 ----------------
            ARREADY <= !ARREADY && ARVALID && !r_pend && !RVALID;
            if (ARVALID && ARREADY) begin raddr <= ARADDR; r_pend <= 1; end
            if (r_pend) begin                       // 延遲一拍再送 R
                r_pend <= 0;
                RVALID <= 1; RLAST <= 1;
                RDATA  <= raddr[31] ? 32'hDEAD_BEEF : mem[raddr[9:2]];
                RRESP  <= raddr[31] ? 2'b10 : 2'b00;
            end
            if (RVALID && RREADY) begin RVALID <= 0; RLAST <= 0; end
        end
    end
endmodule

In [ ]:
%%writefile tb/tb_ahb_slave_wrapper.v
// ============================================================================
// Testbench：AHB Slave Wrapper
// 系統：AHB master BFM（扮演 CPU） -> ahb_slave_wrapper -> wb_slave_model（wb_dma 暫存器）
// 測項：
//   T1 單筆寫入 + 讀回
//   T2 INCR4 burst 寫入 + INCR4 burst 讀回（展示 pipeline 與等待狀態）
//   T3 存取不存在的暫存器 -> 兩拍 ERROR 回應
//   T4 錯誤之後再存取，確認 wrapper 能恢復
// ============================================================================
`timescale 1ns/1ps
module tb_ahb_slave_wrapper;
    reg         HCLK = 0, HRESETn = 0;
    reg         HSEL = 0;
    reg  [31:0] HADDR = 0, HWDATA = 0;
    reg         HWRITE = 0;
    reg  [1:0]  HTRANS = 0;
    reg  [2:0]  HSIZE = 3'b010, HBURST = 0;
    wire        HREADYOUT;
    wire [1:0]  HRESP;
    wire [31:0] HRDATA;
    wire        HREADY = HREADYOUT;     // 單一 slave：匯流排 HREADY 就是它的 HREADYOUT

    wire        wb_cyc, wb_stb, wb_we, wb_ack, wb_err;
    wire [31:0] wb_adr, wb_dat_w, wb_dat_r;
    wire [3:0]  wb_sel;

    always #5 HCLK = ~HCLK;

    ahb_slave_wrapper dut (
        .HCLK(HCLK), .HRESETn(HRESETn),
        .HSEL(HSEL), .HADDR(HADDR), .HWRITE(HWRITE), .HTRANS(HTRANS),
        .HSIZE(HSIZE), .HWDATA(HWDATA), .HREADY(HREADY),
        .HREADYOUT(HREADYOUT), .HRESP(HRESP), .HRDATA(HRDATA),
        .wb_cyc_o(wb_cyc), .wb_stb_o(wb_stb), .wb_we_o(wb_we),
        .wb_adr_o(wb_adr), .wb_sel_o(wb_sel), .wb_dat_o(wb_dat_w),
        .wb_dat_i(wb_dat_r), .wb_ack_i(wb_ack), .wb_err_i(wb_err)
    );

    wb_slave_model #(.WAIT(1)) u_wb (
        .clk(HCLK), .rst(~HRESETn),
        .wb_cyc_i(wb_cyc), .wb_stb_i(wb_stb), .wb_we_i(wb_we),
        .wb_adr_i(wb_adr), .wb_sel_i(wb_sel), .wb_dat_i(wb_dat_w),
        .wb_dat_o(wb_dat_r), .wb_ack_o(wb_ack), .wb_err_o(wb_err)
    );

    // ---------------- AHB master BFM ----------------
    localparam NONSEQ = 2'b10, SEQ = 2'b11, IDLE = 2'b00;
    reg [31:0] wbuf [0:15];
    reg [31:0] rbuf [0:15];
    reg [1:0]  rresp [0:15];
    reg        dp_active, dp_write;
    integer    dp_idx;
    integer    errors = 0;

    // 等待目前的傳輸相位結束（HREADY=1 的那個時脈邊緣），
    // 結束前若有讀取資料相位，順便取回 HRDATA 與 HRESP
    task wait_ready;
        begin
            @(negedge HCLK);
            while (!HREADY) @(negedge HCLK);
            if (dp_active) begin
                rresp[dp_idx] = HRESP;
                if (!dp_write) rbuf[dp_idx] = HRDATA;
            end
            @(posedge HCLK);
        end
    endtask

    // n 拍的 AHB 傳輸：n=1 為 SINGLE，n=4 為 INCR4
    task ahb_xfer;
        input        wr;
        input [31:0] base;
        input integer n;
        integer i;
        begin
            dp_active = 0;
            #1;
            HSEL = 1; HADDR = base; HWRITE = wr; HTRANS = NONSEQ;
            HBURST = (n == 4) ? 3'b011 : 3'b000;
            for (i = 0; i < n; i = i + 1) begin
                wait_ready();           // 第 i 拍的位址相位被接受
                #1;
                dp_active = 1; dp_write = wr; dp_idx = i;
                if (wr) HWDATA = wbuf[i];
                if (i < n - 1) begin HADDR = base + 4 * (i + 1); HTRANS = SEQ; end
                else           begin HTRANS = IDLE; HSEL = 0; end
            end
            wait_ready();               // 最後一拍的資料相位結束
            dp_active = 0;
        end
    endtask

    task check;
        input [31:0] got, exp;
        input [8*24-1:0] name;
        begin
            if (got !== exp) begin
                $display("  [FAIL] %0s : got %h, expected %h", name, got, exp);
                errors = errors + 1;
            end else
                $display("  [PASS] %0s : %h", name, got);
        end
    endtask

    integer k;
    initial begin
        $dumpfile("sim/tb_ahb_slave_wrapper.vcd");
        $dumpvars(0, tb_ahb_slave_wrapper);
        repeat (2) @(posedge HCLK);
        HRESETn = 1;
        @(posedge HCLK);

        $display("== T1 單筆寫入 CSR(0x00) 再讀回 ==");
        wbuf[0] = 32'h0000_0001;
        ahb_xfer(1, 32'h0000_0000, 1);
        ahb_xfer(0, 32'h0000_0000, 1);
        check(rbuf[0], 32'h0000_0001, "T1 read CSR");

        $display("== T2 INCR4 burst 寫入 0x20~0x2C 再 burst 讀回 ==");
        for (k = 0; k < 4; k = k + 1) wbuf[k] = 32'hA5A5_0000 + k;
        ahb_xfer(1, 32'h0000_0020, 4);
        ahb_xfer(0, 32'h0000_0020, 4);
        for (k = 0; k < 4; k = k + 1) check(rbuf[k], 32'hA5A5_0000 + k, "T2 burst read");
        // 直接檢查暫存器內容，確認每一拍寫到正確位址
        for (k = 0; k < 4; k = k + 1) check(u_wb.regs[8 + k], 32'hA5A5_0000 + k, "T2 reg file");

        $display("== T3 讀取不存在的暫存器 0x200 -> 應回 ERROR ==");
        ahb_xfer(0, 32'h0000_0200, 1);
        check({30'd0, rresp[0]}, 32'd1, "T3 HRESP=ERROR");

        $display("== T4 錯誤之後再寫入/讀回 0x04 ==");
        wbuf[0] = 32'h1234_5678;
        ahb_xfer(1, 32'h0000_0004, 1);
        ahb_xfer(0, 32'h0000_0004, 1);
        check(rbuf[0], 32'h1234_5678, "T4 read after error");

        repeat (3) @(posedge HCLK);
        if (errors == 0) $display("\n*** tb_ahb_slave_wrapper : ALL TESTS PASSED ***");
        else             $display("\n*** tb_ahb_slave_wrapper : %0d TEST(S) FAILED ***", errors);
        $finish;
    end

    initial begin #20000; $display("TIMEOUT"); $finish; end
endmodule

In [ ]:
%%writefile tb/tb_ahb_master_wrapper.v
// ============================================================================
// Testbench：AHB Master Wrapper
// 系統：WB master BFM（扮演 wb_dma 的 DMA 引擎） -> ahb_master_wrapper
//       -> AHB 匯流排（含仲裁器，另有一個 CPU master 會搶匯流排）
//       -> ahb_mem_model（記憶體控制器）
// 測項：
//   T1 CPU 正在用匯流排時 DMA 發出寫入 -> wrapper 停在 BUS_REQ 等授權
//   T2 DMA 搬移：從來源區讀 4 個 word，寫到目的區，再比對
//   T3 寫入錯誤區 (0x8000_0000) -> wb_err
//   T4 讀取 RETRY 區 (0x4000_0000) -> wrapper 重新申請匯流排並重送
// ============================================================================
`timescale 1ns/1ps
module tb_ahb_master_wrapper;
    reg HCLK = 0, HRESETn = 0;
    always #5 HCLK = ~HCLK;

    // ---------------- WB master BFM 訊號 ----------------
    reg         wb_cyc = 0, wb_stb = 0, wb_we = 0;
    reg  [31:0] wb_adr = 0, wb_dat_w = 0;
    reg  [3:0]  wb_sel = 4'hF;
    wire [31:0] wb_dat_r;
    wire        wb_ack, wb_err;

    // ---------------- wrapper 的 AHB 訊號 ----------------
    wire        m_HBUSREQ;
    wire [31:0] m_HADDR, m_HWDATA;
    wire [1:0]  m_HTRANS;
    wire        m_HWRITE;
    wire [2:0]  m_HSIZE, m_HBURST;

    // ---------------- 匯流排 ----------------
    wire        HREADY;
    wire [1:0]  HRESP;
    wire [31:0] HRDATA;
    reg         HGRANT_dma = 0;     // 給 wrapper 的授權
    reg         owner_dma  = 0;     // 目前位址匯流排擁有者（1 = DMA，0 = CPU）
    reg         cpu_req    = 0;     // CPU 想用匯流排

    // 仲裁器：CPU 優先；授權只在 HREADY=1 時改變（AHB 規則）
    always @(posedge HCLK) begin
        if (!HRESETn) begin
            HGRANT_dma <= 0; owner_dma <= 0;
        end else if (HREADY) begin
            HGRANT_dma <= m_HBUSREQ & ~cpu_req;
            owner_dma  <= HGRANT_dma;     // 拿到授權的下一拍才擁有位址匯流排
        end
    end

    // 位址/控制多工器（CPU 這裡只佔用匯流排、送 IDLE 傳輸）
    wire [31:0] HADDR  = owner_dma ? m_HADDR  : 32'h0;
    wire [1:0]  HTRANS = owner_dma ? m_HTRANS : 2'b00;
    wire        HWRITE = owner_dma ? m_HWRITE : 1'b0;
    wire [2:0]  HSIZE  = owner_dma ? m_HSIZE  : 3'b010;
    wire [31:0] HWDATA = m_HWDATA;

    ahb_master_wrapper dut (
        .HCLK(HCLK), .HRESETn(HRESETn),
        .HBUSREQ(m_HBUSREQ), .HGRANT(HGRANT_dma), .HREADY(HREADY),
        .HRESP(HRESP), .HRDATA(HRDATA),
        .HADDR(m_HADDR), .HTRANS(m_HTRANS), .HWRITE(m_HWRITE),
        .HSIZE(m_HSIZE), .HBURST(m_HBURST), .HWDATA(m_HWDATA),
        .wb_cyc_i(wb_cyc), .wb_stb_i(wb_stb), .wb_we_i(wb_we),
        .wb_adr_i(wb_adr), .wb_sel_i(wb_sel), .wb_dat_i(wb_dat_w),
        .wb_dat_o(wb_dat_r), .wb_ack_o(wb_ack), .wb_err_o(wb_err)
    );

    ahb_mem_model #(.WAIT(1)) u_mem (
        .HCLK(HCLK), .HRESETn(HRESETn),
        .HSEL(1'b1), .HADDR(HADDR), .HWRITE(HWRITE), .HTRANS(HTRANS),
        .HSIZE(HSIZE), .HWDATA(HWDATA), .HREADY(HREADY),
        .HREADYOUT(HREADY), .HRESP(HRESP), .HRDATA(HRDATA)
    );

    // ---------------- WB master BFM ----------------
    reg [31:0] rd;
    reg        got_err;
    integer    errors = 0;

    task wb_xfer;
        input        we;
        input [31:0] adr, dat;
        begin
            @(posedge HCLK); #1;
            wb_cyc = 1; wb_stb = 1; wb_we = we; wb_adr = adr; wb_dat_w = dat;
            @(negedge HCLK);
            while (!(wb_ack || wb_err)) @(negedge HCLK);
            rd = wb_dat_r; got_err = wb_err;
            @(posedge HCLK); #1;
            wb_cyc = 0; wb_stb = 0; wb_we = 0;
        end
    endtask

    task check;
        input [31:0] got, exp;
        input [8*24-1:0] name;
        begin
            if (got !== exp) begin
                $display("  [FAIL] %0s : got %h, expected %h", name, got, exp);
                errors = errors + 1;
            end else
                $display("  [PASS] %0s : %h", name, got);
        end
    endtask

    integer k, req_cycles;
    // 統計 wrapper 在 BUS_REQ 等了幾拍
    always @(posedge HCLK) if (dut.state == 2'd1) req_cycles = req_cycles + 1;
    // 統計 DATA -> BUS_REQ（⑤ RETRY/SPLIT 重新申請）發生幾次
    integer retry_cnt = 0;
    always @(posedge HCLK) if (dut.state == 2'd3 && dut.next_state == 2'd1) retry_cnt = retry_cnt + 1;

    initial begin
        $dumpfile("sim/tb_ahb_master_wrapper.vcd");
        $dumpvars(0, tb_ahb_master_wrapper);
        // 預先放好 DMA 來源資料
        for (k = 0; k < 4; k = k + 1) u_mem.mem[16 + k] = 32'hC0DE_0000 + k;   // 0x40~0x4C
        req_cycles = 0;
        repeat (2) @(posedge HCLK);
        HRESETn = 1;
        for (k = 0; k < 4; k = k + 1) u_mem.mem[16 + k] = 32'hC0DE_0000 + k;

        $display("== T1 CPU 佔用匯流排時，DMA 寫入 0x00 ==");
        cpu_req = 1;
        fork
            wb_xfer(1, 32'h0000_0000, 32'h1111_2222);
            begin repeat (6) @(posedge HCLK); cpu_req = 0; end   // CPU 6 拍後放手
        join
        check(u_mem.mem[0], 32'h1111_2222, "T1 mem[0x00]");
        if (req_cycles >= 4) $display("  [PASS] T1 wrapper 在 BUS_REQ 等了 %0d 拍", req_cycles);
        else begin $display("  [FAIL] T1 沒有等授權 (%0d 拍)", req_cycles); errors = errors + 1; end

        $display("== T2 DMA 搬移 0x40~0x4C -> 0x80~0x8C ==");
        for (k = 0; k < 4; k = k + 1) begin
            wb_xfer(0, 32'h0000_0040 + 4 * k, 0);
            wb_xfer(1, 32'h0000_0080 + 4 * k, rd);
        end
        for (k = 0; k < 4; k = k + 1) check(u_mem.mem[32 + k], 32'hC0DE_0000 + k, "T2 dst word");

        $display("== T3 寫入錯誤區 0x8000_0000 -> 應收到 wb_err ==");
        wb_xfer(1, 32'h8000_0000, 32'hFFFF_FFFF);
        check({31'd0, got_err}, 32'd1, "T3 wb_err");

        $display("== T4 讀取 RETRY 區 0x4000_0010 -> wrapper 應重送 ==");
        u_mem.mem[4] = 32'h5EED_BEEF;
        wb_xfer(0, 32'h4000_0010, 0);
        check(rd, 32'h5EED_BEEF, "T4 read after RETRY");
        check({31'd0, got_err}, 32'd0, "T4 no error");
        check(retry_cnt, 32'd1, "T4 retry count");

        repeat (3) @(posedge HCLK);
        if (errors == 0) $display("\n*** tb_ahb_master_wrapper : ALL TESTS PASSED ***");
        else             $display("\n*** tb_ahb_master_wrapper : %0d TEST(S) FAILED ***", errors);
        $finish;
    end

    initial begin #20000; $display("TIMEOUT"); $finish; end
endmodule

In [ ]:
%%writefile tb/tb_axi_slave_wrapper.v
// ============================================================================
// Testbench：AXI4 Slave Wrapper
// 系統：AXI master BFM（扮演 CPU） -> axi_slave_wrapper -> wb_slave_model（wb_dma 暫存器）
// 測項：
//   T1 單拍寫入 + 讀回
//   T2 4 拍 INCR burst 寫入 + 4 拍 burst 讀回（讀取時第 2 拍 RREADY 故意晚給）
//   T3 存取不存在的暫存器 -> BRESP / RRESP = SLVERR
//   T4 同一拍同時送 AW 與 AR -> wrapper 應先服務寫入，讀回的是新值
// ============================================================================
`timescale 1ns/1ps
module tb_axi_slave_wrapper;
    reg ACLK = 0, ARESETn = 0;
    always #5 ACLK = ~ACLK;

    reg  [3:0]  AWID = 0, ARID = 0;
    reg  [31:0] AWADDR = 0, ARADDR = 0, WDATA = 0;
    reg  [7:0]  AWLEN = 0, ARLEN = 0;
    reg  [2:0]  AWSIZE = 3'b010, ARSIZE = 3'b010;
    reg  [1:0]  AWBURST = 2'b01, ARBURST = 2'b01;
    reg         AWVALID = 0, WVALID = 0, WLAST = 0, BREADY = 0, ARVALID = 0, RREADY = 0;
    reg  [3:0]  WSTRB = 4'hF;
    wire        AWREADY, WREADY, BVALID, ARREADY, RVALID, RLAST;
    wire [3:0]  BID, RID;
    wire [1:0]  BRESP, RRESP;
    wire [31:0] RDATA;

    wire        wb_cyc, wb_stb, wb_we, wb_ack, wb_err;
    wire [31:0] wb_adr, wb_dat_w, wb_dat_r;
    wire [3:0]  wb_sel;

    axi_slave_wrapper dut (
        .ACLK(ACLK), .ARESETn(ARESETn),
        .AWID(AWID), .AWADDR(AWADDR), .AWLEN(AWLEN), .AWSIZE(AWSIZE), .AWBURST(AWBURST),
        .AWVALID(AWVALID), .AWREADY(AWREADY),
        .WDATA(WDATA), .WSTRB(WSTRB), .WLAST(WLAST), .WVALID(WVALID), .WREADY(WREADY),
        .BID(BID), .BRESP(BRESP), .BVALID(BVALID), .BREADY(BREADY),
        .ARID(ARID), .ARADDR(ARADDR), .ARLEN(ARLEN), .ARSIZE(ARSIZE), .ARBURST(ARBURST),
        .ARVALID(ARVALID), .ARREADY(ARREADY),
        .RID(RID), .RDATA(RDATA), .RRESP(RRESP), .RLAST(RLAST), .RVALID(RVALID), .RREADY(RREADY),
        .wb_cyc_o(wb_cyc), .wb_stb_o(wb_stb), .wb_we_o(wb_we),
        .wb_adr_o(wb_adr), .wb_sel_o(wb_sel), .wb_dat_o(wb_dat_w),
        .wb_dat_i(wb_dat_r), .wb_ack_i(wb_ack), .wb_err_i(wb_err)
    );

    wb_slave_model #(.WAIT(1)) u_wb (
        .clk(ACLK), .rst(~ARESETn),
        .wb_cyc_i(wb_cyc), .wb_stb_i(wb_stb), .wb_we_i(wb_we),
        .wb_adr_i(wb_adr), .wb_sel_i(wb_sel), .wb_dat_i(wb_dat_w),
        .wb_dat_o(wb_dat_r), .wb_ack_o(wb_ack), .wb_err_o(wb_err)
    );

    // ---------------- AXI master BFM ----------------
    reg [31:0] wbuf [0:15];
    reg [31:0] rbuf [0:15];
    reg [1:0]  bresp_got;
    reg [1:0]  rresp_got [0:15];
    reg        rlast_ok;
    integer    errors = 0;
    integer    t_write_done, t_read_addr;   // 記錄時間，用來判斷 T4 的先後順序

    // n 拍寫入：AW 與 W 同時開始送（fork），兩邊都完成後收 B
    task axi_write;
        input [31:0] base;
        input integer n;
        input [3:0] id;
        integer i;
        begin
            @(posedge ACLK); #1;
            fork
                begin   // AW 通道
                    AWVALID = 1; AWADDR = base; AWLEN = n - 1; AWID = id;
                    @(negedge ACLK); while (!AWREADY) @(negedge ACLK);
                    @(posedge ACLK); #1; AWVALID = 0;
                end
                begin   // W 通道
                    for (i = 0; i < n; i = i + 1) begin
                        WVALID = 1; WDATA = wbuf[i]; WLAST = (i == n - 1);
                        @(negedge ACLK); while (!WREADY) @(negedge ACLK);
                        @(posedge ACLK); #1;
                    end
                    WVALID = 0; WLAST = 0;
                end
            join
            BREADY = 1;          // B 通道
            @(negedge ACLK); while (!BVALID) @(negedge ACLK);
            bresp_got = BRESP;
            if (BID !== id) begin $display("  [FAIL] BID mismatch"); errors = errors + 1; end
            @(posedge ACLK); t_write_done = $time; #1; BREADY = 0;
        end
    endtask

    // n 拍讀取；slow_beat >= 0 時，該拍的 RREADY 會晚 6 拍才給（展示 backpressure）
    task axi_read;
        input [31:0] base;
        input integer n;
        input integer slow_beat;
        integer i;
        begin
            @(posedge ACLK); #1;
            ARVALID = 1; ARADDR = base; ARLEN = n - 1; ARID = 4'd5;
            @(negedge ACLK); while (!ARREADY) @(negedge ACLK);
            @(posedge ACLK); t_read_addr = $time; #1; ARVALID = 0;
            rlast_ok = 1;
            for (i = 0; i < n; i = i + 1) begin
                if (i == slow_beat) begin RREADY = 0; repeat (6) @(posedge ACLK); #1; end
                RREADY = 1;
                @(negedge ACLK); while (!RVALID) @(negedge ACLK);
                rbuf[i] = RDATA; rresp_got[i] = RRESP;
                if (RLAST !== (i == n - 1)) rlast_ok = 0;
                @(posedge ACLK); #1;
            end
            RREADY = 0;
        end
    endtask

    task check;
        input [31:0] got, exp;
        input [8*24-1:0] name;
        begin
            if (got !== exp) begin
                $display("  [FAIL] %0s : got %h, expected %h", name, got, exp);
                errors = errors + 1;
            end else
                $display("  [PASS] %0s : %h", name, got);
        end
    endtask

    integer k;
    initial begin
        $dumpfile("sim/tb_axi_slave_wrapper.vcd");
        $dumpvars(0, tb_axi_slave_wrapper);
        repeat (2) @(posedge ACLK);
        ARESETn = 1;

        $display("== T1 單拍寫入 0x08 再讀回 ==");
        wbuf[0] = 32'hCAFE_0008;
        axi_write(32'h08, 1, 4'd1);
        axi_read (32'h08, 1, -1);
        check(rbuf[0], 32'hCAFE_0008, "T1 read");
        check({30'd0, bresp_got}, 0, "T1 BRESP=OKAY");

        $display("== T2 4 拍 burst 寫入 0x40~0x4C，再 burst 讀回 ==");
        for (k = 0; k < 4; k = k + 1) wbuf[k] = 32'hB0B0_0000 + k;
        axi_write(32'h40, 4, 4'd2);
        axi_read (32'h40, 4, 1);
        for (k = 0; k < 4; k = k + 1) check(rbuf[k], 32'hB0B0_0000 + k, "T2 burst read");
        check({31'd0, rlast_ok}, 1, "T2 RLAST only last");
        // 直接檢查暫存器內容，確認 wrapper 有自己遞增位址（不是全寫到同一格）
        for (k = 0; k < 4; k = k + 1) check(u_wb.regs[16 + k], 32'hB0B0_0000 + k, "T2 reg file");

        $display("== T3 存取不存在的暫存器 0x300 -> SLVERR ==");
        wbuf[0] = 32'h0;
        axi_write(32'h300, 1, 4'd3);
        check({30'd0, bresp_got}, 32'd2, "T3 BRESP=SLVERR");
        axi_read(32'h300, 1, -1);
        check({30'd0, rresp_got[0]}, 32'd2, "T3 RRESP=SLVERR");

        $display("== T4 同一拍送 AW 與 AR（位址 0x10）-> 寫入優先 ==");
        wbuf[0] = 32'hF1F1_0010;
        fork
            axi_write(32'h10, 1, 4'd4);
            axi_read (32'h10, 1, -1);
        join
        check(rbuf[0], 32'hF1F1_0010, "T4 read sees new data");
        if (t_read_addr >= t_write_done) $display("  [PASS] T4 讀取在寫入完成後才被接受");
        else begin $display("  [FAIL] T4 順序錯誤"); errors = errors + 1; end

        repeat (3) @(posedge ACLK);
        if (errors == 0) $display("\n*** tb_axi_slave_wrapper : ALL TESTS PASSED ***");
        else             $display("\n*** tb_axi_slave_wrapper : %0d TEST(S) FAILED ***", errors);
        $finish;
    end

    initial begin #20000; $display("TIMEOUT"); $finish; end
endmodule

In [ ]:
%%writefile tb/tb_axi_master_wrapper.v
// ============================================================================
// Testbench：AXI4 Master Wrapper
// 系統：WB master BFM（扮演 wb_dma 的 DMA 引擎） -> axi_master_wrapper
//       -> axi_mem_model（記憶體控制器，故意先收 W 才給 AWREADY）
// 測項：
//   T1 寫入 + 讀回；確認 W 先完成、AW 後完成時 wrapper 不會死結
//   T2 DMA 搬移：從來源區讀 4 個 word，寫到目的區，再比對
//   T3 寫入 / 讀取錯誤區 (0x8000_0000) -> wb_err
// ============================================================================
`timescale 1ns/1ps
module tb_axi_master_wrapper;
    reg ACLK = 0, ARESETn = 0;
    always #5 ACLK = ~ACLK;

    reg         wb_cyc = 0, wb_stb = 0, wb_we = 0;
    reg  [31:0] wb_adr = 0, wb_dat_w = 0;
    reg  [3:0]  wb_sel = 4'hF;
    wire [31:0] wb_dat_r;
    wire        wb_ack, wb_err;

    wire [3:0]  AWID, ARID;
    wire [31:0] AWADDR, ARADDR, WDATA, RDATA;
    wire [7:0]  AWLEN, ARLEN;
    wire [2:0]  AWSIZE, ARSIZE;
    wire [1:0]  AWBURST, ARBURST, BRESP, RRESP;
    wire [3:0]  WSTRB;
    wire        AWVALID, AWREADY, WVALID, WREADY, WLAST, BVALID, BREADY;
    wire        ARVALID, ARREADY, RVALID, RREADY, RLAST;

    axi_master_wrapper dut (
        .ACLK(ACLK), .ARESETn(ARESETn),
        .AWID(AWID), .AWADDR(AWADDR), .AWLEN(AWLEN), .AWSIZE(AWSIZE), .AWBURST(AWBURST),
        .AWVALID(AWVALID), .AWREADY(AWREADY),
        .WDATA(WDATA), .WSTRB(WSTRB), .WLAST(WLAST), .WVALID(WVALID), .WREADY(WREADY),
        .BID(4'd0), .BRESP(BRESP), .BVALID(BVALID), .BREADY(BREADY),
        .ARID(ARID), .ARADDR(ARADDR), .ARLEN(ARLEN), .ARSIZE(ARSIZE), .ARBURST(ARBURST),
        .ARVALID(ARVALID), .ARREADY(ARREADY),
        .RID(4'd0), .RDATA(RDATA), .RRESP(RRESP), .RLAST(RLAST), .RVALID(RVALID), .RREADY(RREADY),
        .wb_cyc_i(wb_cyc), .wb_stb_i(wb_stb), .wb_we_i(wb_we),
        .wb_adr_i(wb_adr), .wb_sel_i(wb_sel), .wb_dat_i(wb_dat_w),
        .wb_dat_o(wb_dat_r), .wb_ack_o(wb_ack), .wb_err_o(wb_err)
    );

    axi_mem_model u_mem (
        .ACLK(ACLK), .ARESETn(ARESETn),
        .AWADDR(AWADDR), .AWVALID(AWVALID), .AWREADY(AWREADY),
        .WDATA(WDATA), .WSTRB(WSTRB), .WVALID(WVALID), .WREADY(WREADY),
        .BRESP(BRESP), .BVALID(BVALID), .BREADY(BREADY),
        .ARADDR(ARADDR), .ARVALID(ARVALID), .ARREADY(ARREADY),
        .RDATA(RDATA), .RRESP(RRESP), .RLAST(RLAST), .RVALID(RVALID), .RREADY(RREADY)
    );

    // ---------------- WB master BFM ----------------
    reg [31:0] rd;
    reg        got_err;
    integer    errors = 0;

    task wb_xfer;
        input        we;
        input [31:0] adr, dat;
        begin
            @(posedge ACLK); #1;
            wb_cyc = 1; wb_stb = 1; wb_we = we; wb_adr = adr; wb_dat_w = dat;
            @(negedge ACLK);
            while (!(wb_ack || wb_err)) @(negedge ACLK);
            rd = wb_dat_r; got_err = wb_err;
            @(posedge ACLK); #1;
            wb_cyc = 0; wb_stb = 0; wb_we = 0;
        end
    endtask

    task check;
        input [31:0] got, exp;
        input [8*24-1:0] name;
        begin
            if (got !== exp) begin
                $display("  [FAIL] %0s : got %h, expected %h", name, got, exp);
                errors = errors + 1;
            end else
                $display("  [PASS] %0s : %h", name, got);
        end
    endtask

    // 統計「W 已完成、AW 還在等」的拍數：證明 AW 與 W 是分開完成的
    integer w_first_cycles = 0;
    always @(posedge ACLK) if (dut.state == 3'd1 && dut.w_done && !dut.aw_done)
        w_first_cycles = w_first_cycles + 1;

    integer k;
    initial begin
        $dumpfile("sim/tb_axi_master_wrapper.vcd");
        $dumpvars(0, tb_axi_master_wrapper);
        repeat (2) @(posedge ACLK);
        ARESETn = 1;
        for (k = 0; k < 4; k = k + 1) u_mem.mem[16 + k] = 32'hD0D0_0000 + k;  // 0x40~0x4C

        $display("== T1 寫入 0x00 再讀回（slave 先收 W 才給 AWREADY） ==");
        wb_xfer(1, 32'h0000_0000, 32'h7777_8888);
        wb_xfer(0, 32'h0000_0000, 0);
        check(rd, 32'h7777_8888, "T1 read back");
        if (w_first_cycles > 0)
            $display("  [PASS] T1 W 先完成、AW 等了 %0d 拍，沒有死結", w_first_cycles);
        else begin $display("  [FAIL] T1 沒有觀察到 W 先完成"); errors = errors + 1; end

        $display("== T2 DMA 搬移 0x40~0x4C -> 0x80~0x8C ==");
        for (k = 0; k < 4; k = k + 1) begin
            wb_xfer(0, 32'h0000_0040 + 4 * k, 0);
            wb_xfer(1, 32'h0000_0080 + 4 * k, rd);
        end
        for (k = 0; k < 4; k = k + 1) check(u_mem.mem[32 + k], 32'hD0D0_0000 + k, "T2 dst word");

        $display("== T3 存取錯誤區 0x8000_0000 -> 應收到 wb_err ==");
        wb_xfer(1, 32'h8000_0000, 32'h0);
        check({31'd0, got_err}, 32'd1, "T3 write wb_err");
        wb_xfer(0, 32'h8000_0000, 32'h0);
        check({31'd0, got_err}, 32'd1, "T3 read wb_err");

        repeat (3) @(posedge ACLK);
        if (errors == 0) $display("\n*** tb_axi_master_wrapper : ALL TESTS PASSED ***");
        else             $display("\n*** tb_axi_master_wrapper : %0d TEST(S) FAILED ***", errors);
        $finish;
    end

    initial begin #20000; $display("TIMEOUT"); $finish; end
endmodule

## 4. 執行模擬（四個 testbench）

In [ ]:
%%bash
cd /content/hw1_bonus
: > sim/sim_log.txt
run() {
  name=$1; shift
  echo "================ $name ================" | tee -a sim/sim_log.txt
  iverilog -g2005 -o sim/$name.out "$@" || { echo "編譯失敗：$name"; exit 1; }
  vvp -n sim/$name.out | grep -v "VCD info" | tee -a sim/sim_log.txt
  echo | tee -a sim/sim_log.txt
}
run ahb_slave  tb/tb_ahb_slave_wrapper.v  rtl/ahb_slave_wrapper.v  tb/wb_slave_model.v
run ahb_master tb/tb_ahb_master_wrapper.v rtl/ahb_master_wrapper.v tb/ahb_mem_model.v
run axi_slave  tb/tb_axi_slave_wrapper.v  rtl/axi_slave_wrapper.v  tb/wb_slave_model.v
run axi_master tb/tb_axi_master_wrapper.v rtl/axi_master_wrapper.v tb/axi_mem_model.v
echo "================ 結果摘要 ================"
grep "\*\*\*" sim/sim_log.txt

## 5. 畫波形圖
從模擬產生的 `sim/*.vcd` 畫出 7 張報告用波形圖（存在 `wave/`）。

In [ ]:
%%writefile scripts/vcd.py
"""極簡 VCD 解析器：讀出每個訊號的 (時間, 值) 變化列表。"""


def parse_vcd(path):
    id2names = {}          # VCD 識別碼 -> [完整階層名稱...]
    scope = []
    changes = {}           # 完整名稱 -> [(time_ps, value_str)]
    t = 0
    in_defs = True
    with open(path) as f:
        for line in f:
            tok = line.split()
            if not tok:
                continue
            if in_defs:
                if tok[0] == "$scope":
                    scope.append(tok[2])
                elif tok[0] == "$upscope":
                    scope.pop()
                elif tok[0] == "$var":
                    vid, name = tok[3], tok[4]
                    full = ".".join(scope + [name])
                    id2names.setdefault(vid, []).append(full)
                    changes[full] = []
                elif tok[0] == "$enddefinitions":
                    in_defs = False
                continue
            c = tok[0]
            if c[0] == "#":
                t = int(c[1:])
            elif c[0] in "bB":
                val, vid = c[1:], tok[1]
                for n in id2names.get(vid, []):
                    changes[n].append((t, val))
            elif c[0] in "01xXzZ":
                val, vid = c[0], c[1:]
                for n in id2names.get(vid, []):
                    changes[n].append((t, val))
    return changes


def value_at(chg, t):
    """訊號在時間 t（ps）的值。"""
    v = "x"
    for tt, vv in chg:
        if tt > t:
            break
        v = vv
    return v


def to_int(v):
    try:
        return int(v, 2)
    except ValueError:
        return None

In [ ]:
%%writefile scripts/plot_waves.py
"""
從 sim/*.vcd 畫出報告用的波形圖（PNG），輸出到 wave/。
用法：在 hw1_bonus 目錄下執行  python3 scripts/plot_waves.py
"""
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager

sys.path.insert(0, os.path.dirname(__file__))
from vcd import parse_vcd, value_at, to_int  # noqa: E402

ROOT = os.path.join(os.path.dirname(__file__), "..")

# 中文字型（找不到就用預設字型）
_names = sorted({f.name for f in font_manager.fontManager.ttflist})
for fam in ["Noto Sans CJK TC", "Noto Sans TC", "Microsoft JhengHei", "PingFang TC",
            "Noto Sans CJK JP", "Noto Sans CJK SC", "Noto Sans CJK KR"]:
    if fam in _names:
        plt.rcParams["font.family"] = fam
        break
else:   # 最後手段：任何名稱含 CJK 的字型
    _cjk = [n for n in _names if "CJK" in n]
    if _cjk:
        plt.rcParams["font.family"] = _cjk[0]

C_LINE = "#1f4e79"
C_BUS = "#e8f0f8"
C_STATE = "#fdf0e6"
C_STATE_EDGE = "#b5541c"
C_X = "#dddddd"
C_BAND = ["#eef6ee", "#f6eef6", "#eef2f8", "#f8f4e8"]


def fmt_bus(v, width):
    n = to_int(v)
    if n is None:
        return None
    if width <= 4:
        return str(n)
    return f"{n:X}"


def draw_bit(ax, chg, y, t0, t1, h=0.6):
    pts_t, pts_v = [t0], [value_at(chg, t0 * 1000)]
    for tt, vv in chg:
        tns = tt / 1000
        if t0 < tns < t1:
            pts_t.append(tns)
            pts_v.append(vv)
    pts_t.append(t1)
    for i in range(len(pts_v)):
        a, b = pts_t[i], pts_t[i + 1]
        v = pts_v[i]
        if v in "xXzZ":
            ax.fill_between([a, b], y, y + h, color=C_X, lw=0)
            continue
        yy = y + (h if v == "1" else 0)
        ax.plot([a, b], [yy, yy], color=C_LINE, lw=1.4)
        if i + 1 < len(pts_v) and pts_v[i + 1] not in "xXzZ" and pts_v[i + 1] != v:
            ax.plot([b, b], [y, y + h], color=C_LINE, lw=1.4)


def draw_bus(ax, chg, y, t0, t1, width, names=None, h=0.6, fill=C_BUS, edge=C_LINE):
    segs = []
    cur = value_at(chg, t0 * 1000)
    start = t0
    for tt, vv in chg:
        tns = tt / 1000
        if t0 < tns < t1 and vv != cur:
            segs.append((start, tns, cur))
            start, cur = tns, vv
    segs.append((start, t1, cur))
    sl = min(1.2, (t1 - t0) / 250)       # 斜邊寬度
    for a, b, v in segs:
        n = to_int(v)
        if n is None:
            ax.fill_between([a, b], y, y + h, color=C_X, lw=0)
            continue
        poly_x = [a, a + sl, b - sl, b, b - sl, a + sl, a]
        poly_y = [y + h / 2, y + h, y + h, y + h / 2, y, y, y + h / 2]
        ax.fill(poly_x, poly_y, color=fill, ec=edge, lw=0.9)
        label = names.get(n, str(n)) if names else fmt_bus(v, width)
        span = b - a
        if label is not None and span > len(label) * (t1 - t0) / 230:
            ax.text((a + b) / 2, y + h / 2, label, ha="center", va="center", fontsize=7.5)


def plot(vcd_file, out_png, title, t0, t1, rows, bands=(), period=10):
    chg = parse_vcd(os.path.join(ROOT, vcd_file))
    n = len(rows)
    fig_h = 0.42 * n + 1.3
    fig, ax = plt.subplots(figsize=(14, fig_h))
    labels = []
    for i, row in enumerate(rows):
        label, sig, kind = row[0], row[1], row[2]
        y = (n - 1 - i) * 1.0
        c = chg[sig]
        if kind == "bit":
            draw_bit(ax, c, y, t0, t1)
        elif kind == "bus":
            draw_bus(ax, c, y, t0, t1, row[3] if len(row) > 3 else 32)
        elif kind == "state":
            draw_bus(ax, c, y, t0, t1, 8, names=row[3], fill=C_STATE, edge=C_STATE_EDGE)
        labels.append((y + 0.3, label))
    # 時脈邊緣格線
    t = (int(t0 // period) + 1) * period - period / 2
    while t < t1:
        if t > t0:
            ax.axvline(t, color="#c8c8c8", lw=0.5, ls=":", zorder=0)
        t += period
    # 測項區段底色與標籤
    for k, (a, b, txt) in enumerate(bands):
        ax.axvspan(a, b, color=C_BAND[k % len(C_BAND)], zorder=-1)
        ax.text((a + b) / 2, n - 0.15, txt, ha="center", va="bottom", fontsize=9, color="#333")
    ax.set_yticks([p for p, _ in labels])
    ax.set_yticklabels([l for _, l in labels], fontsize=8.5, family="monospace")
    ax.set_xlim(t0, t1)
    ax.set_ylim(-0.4, n + 0.35)
    ax.set_xlabel("time (ns)")
    ax.set_title(title, fontsize=12, loc="left")
    for s in ["top", "right", "left"]:
        ax.spines[s].set_visible(False)
    ax.tick_params(axis="y", length=0)
    fig.tight_layout()
    fig.savefig(os.path.join(ROOT, out_png), dpi=150)
    plt.close(fig)
    print("wrote", out_png)


# ---------------------------------------------------------------------------
AHBS = {0: "IDLE", 1: "WB_ACCESS", 2: "ERR1", 3: "ERR2"}
AHBM = {0: "IDLE", 1: "BUS_REQ", 2: "ADDR", 3: "DATA"}
AXIS = {0: "IDLE", 1: "W_DATA", 2: "W_RESP", 3: "R_WB", 4: "R_DATA"}
AXIM = {0: "IDLE", 1: "W_SEND", 2: "W_RESP", 3: "R_ADDR", 4: "R_DATA"}
HTR = {0: "IDLE", 1: "BUSY", 2: "NONSEQ", 3: "SEQ"}
HRS = {0: "OKAY", 1: "ERROR", 2: "RETRY", 3: "SPLIT"}


def main():
    os.makedirs(os.path.join(ROOT, "wave"), exist_ok=True)

    # ---------------- AHB slave ----------------
    p = "tb_ahb_slave_wrapper."
    rows = [
        ("HCLK", p + "HCLK", "bit"),
        ("HSEL", p + "HSEL", "bit"),
        ("HTRANS", p + "HTRANS", "state", HTR),
        ("HADDR", p + "HADDR", "bus"),
        ("HWRITE", p + "HWRITE", "bit"),
        ("HWDATA", p + "HWDATA", "bus"),
        ("HREADYOUT", p + "HREADYOUT", "bit"),
        ("HRESP", p + "HRESP", "state", HRS),
        ("HRDATA", p + "HRDATA", "bus"),
        ("FSM state", p + "dut.state", "state", AHBS),
        ("wb_stb_o", p + "wb_stb", "bit"),
        ("wb_adr_o", p + "wb_adr", "bus"),
        ("wb_ack_i", p + "wb_ack", "bit"),
        ("wb_err_i", p + "wb_err", "bit"),
    ]
    plot("sim/tb_ahb_slave_wrapper.vcd", "wave/1a_ahb_slave_burst.png",
         "AHB Slave Wrapper：INCR4 burst 寫入與讀取（每拍插入等待狀態）",
         122, 492, rows, bands=[(125, 290, "T2 burst 寫入 0x20~0x2C"),
                                (290, 490, "T2 burst 讀取 0x20~0x2C")])
    plot("sim/tb_ahb_slave_wrapper.vcd", "wave/1b_ahb_slave_error.png",
         "AHB Slave Wrapper：兩拍 ERROR 回應，之後恢復正常",
         462, 632, rows, bands=[(465, 530, "T3 讀不存在的暫存器"),
                                (530, 630, "T4 錯誤後再寫入/讀回")])

    # ---------------- AHB master ----------------
    p = "tb_ahb_master_wrapper."
    rows = [
        ("HCLK", p + "HCLK", "bit"),
        ("wb_stb_i", p + "wb_stb", "bit"),
        ("wb_we_i", p + "wb_we", "bit"),
        ("wb_adr_i", p + "wb_adr", "bus"),
        ("cpu_req", p + "cpu_req", "bit"),
        ("HBUSREQ", p + "m_HBUSREQ", "bit"),
        ("HGRANT", p + "HGRANT_dma", "bit"),
        ("FSM state", p + "dut.state", "state", AHBM),
        ("HTRANS", p + "HTRANS", "state", HTR),
        ("HADDR", p + "HADDR", "bus"),
        ("HREADY", p + "HREADY", "bit"),
        ("HRESP", p + "HRESP", "state", HRS),
        ("HRDATA", p + "HRDATA", "bus"),
        ("wb_ack_o", p + "wb_ack", "bit"),
        ("wb_err_o", p + "wb_err", "bit"),
    ]
    plot("sim/tb_ahb_master_wrapper.vcd", "wave/2a_ahb_master_arbitration.png",
         "AHB Master Wrapper：CPU 佔用匯流排時停在 BUS_REQ，取得 HGRANT 後才送位址",
         22, 202, rows, bands=[(25, 130, "T1 等待仲裁 -> 寫入 0x00"),
                               (130, 200, "T2 DMA 讀取來源")])
    plot("sim/tb_ahb_master_wrapper.vcd", "wave/2b_ahb_master_retry.png",
         "AHB Master Wrapper：寫入錯誤區收到 ERROR；讀取時收到 RETRY 後重新申請匯流排並重送",
         692, 892, rows, bands=[(695, 765, "T3 寫入錯誤區 -> wb_err"),
                                (765, 890, "T4 RETRY -> 重新申請並重送")])

    # ---------------- AXI slave ----------------
    p = "tb_axi_slave_wrapper."
    rows = [
        ("ACLK", p + "ACLK", "bit"),
        ("AWVALID", p + "AWVALID", "bit"),
        ("AWREADY", p + "AWREADY", "bit"),
        ("AWADDR", p + "AWADDR", "bus"),
        ("AWLEN", p + "AWLEN", "bus", 4),
        ("WVALID", p + "WVALID", "bit"),
        ("WREADY", p + "WREADY", "bit"),
        ("WLAST", p + "WLAST", "bit"),
        ("WDATA", p + "WDATA", "bus"),
        ("BVALID", p + "BVALID", "bit"),
        ("BREADY", p + "BREADY", "bit"),
        ("ARVALID", p + "ARVALID", "bit"),
        ("ARREADY", p + "ARREADY", "bit"),
        ("RVALID", p + "RVALID", "bit"),
        ("RREADY", p + "RREADY", "bit"),
        ("RLAST", p + "RLAST", "bit"),
        ("RDATA", p + "RDATA", "bus"),
        ("FSM state", p + "dut.state", "state", AXIS),
        ("wb_stb_o", p + "wb_stb", "bit"),
        ("wb_adr_o", p + "wb_adr", "bus"),
        ("wb_ack_i", p + "wb_ack", "bit"),
    ]
    plot("sim/tb_axi_slave_wrapper.vcd", "wave/3a_axi_slave_burst.png",
         "AXI Slave Wrapper：4 拍 burst 寫入與讀取（wrapper 自己遞增位址；讀取第 2 拍 RREADY 晚給，RVALID 保持等待）",
         142, 502, rows, bands=[(145, 295, "T2 burst 寫入 (AWLEN=3)"),
                                (295, 500, "T2 burst 讀取 (ARLEN=3)")])
    plot("sim/tb_axi_slave_wrapper.vcd", "wave/3b_axi_slave_priority.png",
         "AXI Slave Wrapper：AW 與 AR 同一拍到達，寫入優先，讀取等寫入完成後才被接受",
         622, 732, rows, bands=[(625, 680, "T4 寫入先服務"), (680, 730, "接著才服務讀取")])

    # ---------------- AXI master ----------------
    p = "tb_axi_master_wrapper."
    rows = [
        ("ACLK", p + "ACLK", "bit"),
        ("wb_stb_i", p + "wb_stb", "bit"),
        ("wb_we_i", p + "wb_we", "bit"),
        ("wb_adr_i", p + "wb_adr", "bus"),
        ("FSM state", p + "dut.state", "state", AXIM),
        ("AWVALID", p + "AWVALID", "bit"),
        ("AWREADY", p + "AWREADY", "bit"),
        ("WVALID", p + "WVALID", "bit"),
        ("WREADY", p + "WREADY", "bit"),
        ("aw_done", p + "dut.aw_done", "bit"),
        ("w_done", p + "dut.w_done", "bit"),
        ("BVALID", p + "BVALID", "bit"),
        ("BREADY", p + "BREADY", "bit"),
        ("ARVALID", p + "ARVALID", "bit"),
        ("ARREADY", p + "ARREADY", "bit"),
        ("RVALID", p + "RVALID", "bit"),
        ("RREADY", p + "RREADY", "bit"),
        ("RDATA", p + "RDATA", "bus"),
        ("wb_ack_o", p + "wb_ack", "bit"),
    ]
    plot("sim/tb_axi_master_wrapper.vcd", "wave/4a_axi_master_write_read.png",
         "AXI Master Wrapper：AW 與 W 同時送出；slave 先收 W、後給 AWREADY，wrapper 用旗標記錄，不會死結",
         22, 182, rows, bands=[(25, 115, "T1 寫入 0x00"), (115, 180, "T1 讀回 0x00")])


if __name__ == "__main__":
    main()

In [ ]:
!python3 scripts/plot_waves.py 2>&1 | grep -v -i "glyph"

from IPython.display import Image, display, Markdown
import glob
for png in sorted(glob.glob('wave/*.png')):
    display(Markdown(f'**{png}**'))
    display(Image(png, width=1100))

## 6. 反例實驗：AXI 死結（報告加分點）
把 AXI master wrapper 改成「等 AWREADY 才送 WVALID」，同一個 testbench 會在第一筆寫入就 **TIMEOUT**，
證明 AXI 規範要求 AW 與 W 不能互相等待。

In [ ]:
%%bash
cd /content/hw1_bonus
mkdir -p /tmp/naive && cd /tmp/naive && mkdir -p sim
sed 's/assign WVALID  = (state == S_W_SEND) \& ~w_done;/assign WVALID  = (state == S_W_SEND) \& aw_done \& ~w_done;/' \
    /content/hw1_bonus/rtl/axi_master_wrapper.v > naive_axi_master.v
echo "修改後的那一行："
grep -n "assign WVALID" naive_axi_master.v
echo
iverilog -g2005 -o naive.out /content/hw1_bonus/tb/tb_axi_master_wrapper.v naive_axi_master.v /content/hw1_bonus/tb/axi_mem_model.v
vvp -n naive.out | grep -v "VCD info"

## 7. 下載結果

In [ ]:
!cd /content && rm -f hw1_bonus_results.zip && zip -rq hw1_bonus_results.zip hw1_bonus -x "hw1_bonus/sim/*.out"
!ls -lh /content/hw1_bonus_results.zip
from google.colab import files
files.download('/content/hw1_bonus_results.zip')